In [2]:
import os
import re
import glob
from datetime import datetime
from pathlib import Path

import pdfplumber
import pandas as pd



In [139]:
import re
import glob
from datetime import datetime
from pathlib import Path
from typing import Optional

import pdfplumber
import pandas as pd


LOC_MONTHS = [
    "Jan.", "Feb.", "Mar.", "Apr.", "May.", "Jun.",
    "Jul.", "Aug.", "Sep.", "Oct.", "Nov.", "Dec."
]
CK_MONTHS = [
    "Jan", "Feb", "Mar", "Apr", "May", "Jun",
    "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"
]

TD_MONTH_RE = r"(jan|fev|fév|mar|avr|mai|jun|jui|aou|aoû|sep|oct|nov|dec|déc)"


LOC_LINE_RE = re.compile(
    r"""^\s*['"]?
    (?P<tmon>Jan\.|Feb\.|Mar\.|Apr\.|May\.|Jun\.|Jul\.|Aug\.|Sep\.|Oct\.|Nov\.|Dec\.)\s*(?P<tday>\d{1,2})\s*
    (?P<pmon>Jan\.|Feb\.|Mar\.|Apr\.|May\.|Jun\.|Jul\.|Aug\.|Sep\.|Oct\.|Nov\.|Dec\.)\s*(?P<pday>\d{1,2})\s+
    (?P<desc>.*?)
    \s+(?P<amt>-?\d[\d,]*\.\d{2})
    (?:\s+(?P<cr>CR))?
    \s*['"]?\s*$
    """,
    re.VERBOSE
)


CK_LINE_RE = re.compile(
    r"""^\s*['"]?
    (?P<mon>Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)
    (?P<day>\d{2})\s+
    (?P<desc>.*?)
    \s+(?P<amt>-?\d[\d,]*\.\d{2})
    \s+(?P<bal>-?\d[\d,]*\.\d{2})
    \s*['"]?\s*$
    """,
    re.VERBOSE
)

TANGERINE_LINE_RE = re.compile(
    r"""^\s*['"]?
    (?P<day>\d{2})\s+
    (?P<mon>Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)\s+
    (?P<year>\d{4})\s+
    (?P<desc>.*?)
    \s+(?P<amt>\d[\d,]*\.\d{2})
    \s+(?P<bal>\d[\d,]*\.\d{2})
    \s*['"]?\s*$
    """,
    re.VERBOSE
)
TANGERINE_SPLIT_LINE_RE = re.compile(
    r"""^\s*['"]?
    (?P<day>\d{2})\s+
    (?P<mon>Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)\s+
    (?P<year>\d{4})
    \s+(?P<amt>-?\d[\d,]*\.\d{2})
    \s+(?P<bal>-?\d[\d,]*\.\d{2})
    \s*['"]?\s*$
    """,
    re.VERBOSE
)
TD_FIRST_AMOUNT_RE = re.compile(
    r"""(?P<sign>-)?(?P<amt>\d[\d\s]*,\d{2})\s*\$?""",
    re.VERBOSE
)

# # Transaction start: "19mar 20mar ..." OR "19 mar 20 mar ..."
# TD_PREFIX_RE = re.compile(
#     rf"""^\s*['"]?
#     (?P<tday>\d{{1,2}})\s*(?P<tmon>{TD_MONTH_RE})\s+
#     (?P<pday>\d{{1,2}})\s*(?P<pmon>{TD_MONTH_RE})\s+
#     (?P<rest>.+?)\s*['"]?\s*$
#     """,
#     re.IGNORECASE | re.VERBOSE
# )

# Allow day+month with optional space: "19mar" or "19 mar"
TD_DATE_RE = rf"\d{{1,2}}\s*{TD_MONTH_RE}"

# Must start with two date tokens (operation date + posting date)
TD_PREFIX_RE = re.compile(rf"^\s*{TD_DATE_RE}\s+{TD_DATE_RE}\b", re.IGNORECASE)

# Amount like "22,96" optionally with "$", and thousands separated by spaces: "10 626,13" :contentReference[oaicite:4]{index=4}
TD_AMOUNT_RE = re.compile(r"-?\d[\d\s]*,\d{2}\$?", re.IGNORECASE)


# remove leading noise tokens until we hit a date token
LEADING_NOISE_RE = re.compile(rf"^\s*(?:[A-Za-z]|_|\d+)\s+(?={TD_DATE_RE}\b)", re.IGNORECASE)

TD_LINE_RE = re.compile(
    rf"""^\s*['"]?
    (?P<tday>\d{{1,2}})\s*(?P<tmon>{TD_MONTH_RE})\s+
    (?P<pday>\d{{1,2}})\s*(?P<pmon>{TD_MONTH_RE})\s+
    (?P<desc>.*?)
    \s+(?P<sign>-)?(?P<amt>\d[\d\s]*,\d{{2}})\s*\$?
    (?:\s+.*)?   # allow redundant text after amount (e.g., 'Frais 0,00$' or 'S')
    \s*['"]?\s*$
    """,
    re.IGNORECASE | re.VERBOSE
)



def _td_amount_to_float(sign: Optional[str], amt_fr: str) -> float:
    """
    Convert '1 999,99' -> 1999.99 and apply sign if '-' exists.
    """
    x = amt_fr.replace(" ", "").replace("\u00A0", "").replace(",", ".")
    val = float(x)
    if sign:  # '-' directly before the amount in TD statement
        val = -val
    return val

def _to_float(x: str) -> float:
    return float(x.replace(",", "").strip())

def is_tangerine_transaction_line(line: str):
    if not line:
        return False
    s = line.strip().strip("'").strip('"').strip()
    if bool(TANGERINE_LINE_RE.match(s)):
        return 'Normal'
    if  bool(TANGERINE_SPLIT_LINE_RE.match(s)):
        return 'Split'
    
    return False


def looks_like_td_amount_only_line(s: str) -> bool:
    # e.g. "1avr 3avr 60,00$" (no real description words)
    if not (TD_PREFIX_RE.search(s) and TD_AMOUNT_RE.search(s)):
        return False
    # if after removing dates+amount there's almost nothing, treat as split head
    tmp = TD_PREFIX_RE.sub("", s).strip()
    tmp = TD_AMOUNT_RE.sub("", tmp).strip()
    return len(tmp) <= 2  # very short => likely split

def is_td_transaction_line(line: str) -> bool:
    if not line:
        return False
    s = line.strip().strip("'").strip('"').strip()
    return bool(TD_PREFIX_RE.search(s) and TD_AMOUNT_RE.search(s))

def clean_td_line(line: str) -> str:
    if not line:
        return ""
    s = line.strip().strip("'").strip('"').strip()

    # repeatedly strip leading noise like "T " or "5 " or "_" before the first date
    while True:
        new_s = re.sub(LEADING_NOISE_RE, "", s)
        if new_s == s:
            break
        s = new_s.strip()

    # collapse whitespace
    s = re.sub(r"\s+", " ", s).strip()
    return s

def merge_split_td_transaction(amount_line: str, desc_line: str) -> Optional[str]:
    """
    amount_line example: "2 1avr 3avr 60,00$"
    desc_line   example: "8 CHRONO-RECHARGEOPUSMONTREAL"

    returns: "1avr 3avr CHRONO-RECHARGEOPUSMONTREAL 60,00$"
    """
    a = amount_line
    d = desc_line

    if not (TD_PREFIX_RE.search(a) and TD_AMOUNT_RE.search(a)):
        return None

    # Extract the first amount from amount_line
    m_amt = TD_AMOUNT_RE.search(a)
    amt = m_amt.group(0).strip()

    # Take just the "date date" prefix from amount_line (everything before description)
    # Example a: "1avr 3avr 60,00$" -> prefix="1avr 3avr"
    parts = a.split()
    if len(parts) < 2:
        return None
    prefix = f"{parts[0]} {parts[1]}"

    # Remove leading junk digits/underscores from desc
    d = re.sub(r"^(?:\d+|_)+\s*", "", d).strip()

    return f"{prefix} {d} {amt}"


def clean_split_tangerine_transaction(split_tail_line: str, description: str) -> Optional[str]:
    """
    Convert a split transaction into a normal one.

    Inputs:
      - split_tail_line: e.g. "01 Apr 2025 44.83 1,638.75"
      - description: e.g. "Interac - Purchase - KIM PHAT #3733 MONTREAL, QC, CA"

    Output:
      - normal line: "01 Apr 2025 <description> 44.83 1,638.75"
      - returns None if split_tail_line doesn't match the expected format
    """
    if not split_tail_line or not description:
        return None

    s = split_tail_line.strip().strip("'").strip('"').strip()
    desc = description.strip().strip("'").strip('"').strip()

    m = TANGERINE_SPLIT_LINE_RE.match(s)
    if not m:
        return None

    # Normalize whitespace in description
    desc = re.sub(r"\s+", " ", desc).strip()

    return (
        f"{m.group('day')} {m.group('mon')} {m.group('year')} "
        f"{desc} {m.group('amt')} {m.group('bal')}"
    )

def words_to_lines(words, y_tol=2.0):
    """
    Group extracted words into lines using their y-position ('top').
    """
    if not words:
        return []

    # Sort words by y then x
    words = sorted(words, key=lambda w: (w["top"], w["x0"]))

    lines = []
    current = []
    current_top = words[0]["top"]

    for w in words:
        if abs(w["top"] - current_top) <= y_tol:
            current.append(w)
        else:
            # finalize current line
            current_sorted = sorted(current, key=lambda ww: ww["x0"])
            line_text = " ".join(ww["text"] for ww in current_sorted).strip()
            if line_text:
                lines.append(line_text)
            current = [w]
            current_top = w["top"]

    # finalize last
    current_sorted = sorted(current, key=lambda ww: ww["x0"])
    line_text = " ".join(ww["text"] for ww in current_sorted).strip()
    if line_text:
        lines.append(line_text)

    return lines


def parse_LOC_MC_transaction_block(line: str) -> dict:
    """
    Parse one transaction line like:
      Apr. 23 Apr. 24 PAYMENT RECEIVED - THANK YOU 555.55 CR
      Apr. 2 Apr. 3 USD 65.99@1.477648128 OPENAI *CHATGPT 97.51

    Rule:
      - If it ends with amount only => debit
      - If it ends with amount + 'CR' => credit
    Returns dict with trans date, posting date, description, amount, card type
    """
    # Split first 4 tokens: Mon. DD Mon. DD
    if not line:
        return None

    s = line.strip()
    # strip wrapping quotes if present (common when printing lists)
    if (s.startswith("'") and s.endswith("'")) or (s.startswith('"') and s.endswith('"')):
        s = s[1:-1].strip()

    m = LOC_LINE_RE.match(s)
    if not m:
        return None

    amt = float(m.group("amt").replace(",", ""))
    is_credit = m.group("cr") is not None

    return {
        "TRANS DATE": f"{m.group('tmon')} {m.group('tday')}",
        "POSTING DATE": f"{m.group('pmon')} {m.group('pday')}",
        "DESCRIPTION": m.group("desc").strip(),
        "AMOUNT": amt,
        "CARD TYPE": "credit" if is_credit else "debit",
    }


def parse_CK_Tangerine_transaction_block(line: str, balance: float, type_file: str) -> dict:
    """
    Parse one transaction line for CK or Tangerine.

    CK example:
      Apr03 INTERAC e-Transfer Received 1,999.99 2,115.62

    Tangerine examples:
      01 Apr 2025 EFT Withdrawal to RBCINS-LIFE 239.00 1,683.58

    Card type rule (based on balance movement vs prev_balance):
      - if new_balance > prev_balance => credit
      - else => debit

    Returns a tuple (dict with trans date, description, amount, card type, new_balance)
    """
    if not line:
        return None, balance

    s = line.strip()
    # strip wrapping quotes if present (common when printing lists)
    if (s.startswith("'") and s.endswith("'")) or (s.startswith('"') and s.endswith('"')):
        s = s[1:-1].strip()
    s = s.strip().strip("'").strip('"').strip()

    if type_file == "Tangerine":
        m = TANGERINE_LINE_RE.match(s)
    elif type_file == "CK":
        m = CK_LINE_RE.match(s)
        
    if not m:
        return None, balance

    amt = _to_float(m.group("amt"))
    new_balance = _to_float(m.group("bal"))

    is_credit = new_balance > balance

    return {
        "TRANS DATE": f"{m.group('mon')} {m.group('day')}",
        "DESCRIPTION": m.group("desc").strip(),
        "AMOUNT": amt,
        "CARD TYPE": "credit" if is_credit else "debit",
    }, new_balance

def parse_TD_transaction_block(line: str) -> dict:
    """
    TD French credit card transaction parser.

    Handles:
      - normal: '19mar 20mar KERNELSANJOUANJOU 22,96$'
      - redundant text after amount
      - trailing 'S' after amount
      - extra 'Frais 0,00$' after amount (we keep the FIRST amount)
      - spaces or no spaces in month tokens (e.g., '19 jan' vs '19jan') :contentReference[oaicite:2]{index=2}

    Returns:
      dict with trans date, posting date, description, amount, card type (always 'credit' for TD)
    """
    if not line:
        return None

    s = line.strip()

    # strip wrapping quotes if present
    if (s.startswith("'") and s.endswith("'")) or (s.startswith('"') and s.endswith('"')):
        s = s[1:-1].strip()
    s = s.strip().strip("'").strip('"').strip()

    m = TD_LINE_RE.match(s)
    if not m:
        return None

    amount = _td_amount_to_float(m.group("sign"), m.group("amt"))

    desc = m.group("desc").strip()
    desc = re.sub(r"\s+", " ", desc).strip()
    desc = desc.rstrip(" ,;:-")


    return {
        "TRANS DATE": f"{int(m.group('tday'))}{m.group('tmon')}",
        "POSTING DATE": f"{int(m.group('pday'))}{m.group('pmon')}",
        "DESCRIPTION": desc,
        "AMOUNT": amount,
        "CARD TYPE": "credit" # TD file is all credit-card type (your note)
    }

def extract_LOC_MC_transactions_from_pdf(pdf_path: str, file_type: str) -> list[dict]:
    rows = []
    with pdfplumber.open(pdf_path) as pdf:
        for page_no, page in enumerate(pdf.pages):
            if (file_type == "MC") or (file_type == "LOC" and page_no == len(pdf.pages) - 1):
                words = page.extract_words(use_text_flow=True, keep_blank_chars=False)
                lines = words_to_lines(words, y_tol=2.0)
                data_lines = []
                for line in lines:
                    for month in LOC_MONTHS:
                        if line.startswith(month):
                            data_lines.append(line.strip())
                            break
                # Build transaction blocks: a block starts when line begins with "Mon. DD Mon. DD"
                for line in data_lines:
                        rows.append(parse_LOC_MC_transaction_block(line))
    return rows

def extract_CK_Tangerine_transactions_from_pdf(pdf_path: str, type_file: str) -> list[dict]:
    rows = []
    with pdfplumber.open(pdf_path) as pdf:
        balance = 0
        for page_no, page in enumerate(pdf.pages):
            words = page.extract_words(use_text_flow=True, keep_blank_chars=False)
            lines = words_to_lines(words, y_tol=2.0)
            data_lines = []
            for id, line in enumerate(lines):
                if "Openingbalance" in line or "Opening Balance" in line:
                    balance = _to_float(line.split()[-1])
                    continue    
                if "Closingtotals" in line or "Closing Balance" in line:
                    break

                if type_file == "Tangerine":
                    case = is_tangerine_transaction_line(line)
                    if case == "Normal":
                        data_lines.append(line.strip())
                    elif case == "Split" and id > 0:  # split line should not be the first line of the page
                        cleaned = clean_split_tangerine_transaction(line, lines[id-1])
                        if cleaned:
                            data_lines.append(cleaned)
                    
                elif type_file == "CK":
                    for month in CK_MONTHS:
                        if line.startswith(month):
                            data_lines.append(line.strip())
                            break

            # Build transaction blocks: a block starts when line begins with "Mon DD"
            for line in data_lines:
                    transaction, balance = parse_CK_Tangerine_transaction_block(line, balance, type_file)
                    rows.append(transaction)
    return rows 


def extract_TD_transactions_from_pdf(pdf_path: str) -> list[dict]:
    rows = []
    # amount = 0
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:                
                words = page.extract_words(use_text_flow=True, keep_blank_chars=False)
                lines = words_to_lines(words, y_tol=2.0)
                data_lines = []
                
                for id, line in enumerate(lines):
                    # print(line.strip())
                    line_strip = clean_td_line(line.strip())
                    
                    if is_td_transaction_line(line_strip):
                        if looks_like_td_amount_only_line(line_strip):
                            # likely split line, try to merge with previous line
                            merged = merge_split_td_transaction(line_strip, clean_td_line(lines[id+1]) if id+1 < len(lines) else "")
                            if merged:
                                data_lines.append(merged)  # replace previous line with merged result
                                # print("Merged split line:", data_lines[-1])
                                continue  # skip adding the next line since it's merged
                            # else:
                            #     print("Could not merge split line, added as is:", data_lines[-1])
                        else:
                            data_lines.append(line_strip)
                        # print(data_lines[-1])
                for line in data_lines:
                    transaction = parse_TD_transaction_block(line)
                    if transaction:
                        rows.append(transaction)
                        # amount += transaction["AMOUNT"]
    # print(f"Total amount from TD transactions: {amount}")
    return rows

def extract_LOC_folder_to_csv(root_path: str, output_csv: str):
    all_rows = []
    for file in os.listdir(root_path):
        for pdf_file in glob.glob(os.path.join(root_path, file)):
            all_rows.extend(extract_LOC_MC_transactions_from_pdf(pdf_file, "LOC"))

    df = pd.DataFrame(all_rows, columns=["TRANS DATE", "POSTING DATE", "DESCRIPTION", "AMOUNT", "CARD TYPE"])
    df.to_csv(output_csv, index=False)
    print(f"Wrote {len(df)} rows to {output_csv}")

def extract_MC_folder_to_csv(root_path: str, output_csv: str):
    all_rows = []
    for file in os.listdir(root_path):
        for pdf_file in glob.glob(os.path.join(root_path, file)):
            all_rows.extend(extract_LOC_MC_transactions_from_pdf(pdf_file, "MC"))

    df = pd.DataFrame(all_rows, columns=["TRANS DATE", "POSTING DATE", "DESCRIPTION", "AMOUNT", "CARD TYPE"])
    df.to_csv(output_csv, index=False)
    print(f"Wrote {len(df)} rows to {output_csv}")

def extract_CK_folder_to_csv(root_path: str, output_csv: str):
    all_rows = []
    for file in os.listdir(root_path):
        for pdf_file in glob.glob(os.path.join(root_path, file)):
            try:
                all_rows.extend(extract_CK_Tangerine_transactions_from_pdf(pdf_file, "CK"))
            except Exception:
                continue

    df = pd.DataFrame(all_rows, columns=["TRANS DATE", "DESCRIPTION", "AMOUNT", "CARD TYPE"])
    df.to_csv(output_csv, index=False)
    print(f"Wrote {len(df)} rows to {output_csv}")


def extract_Tangerine_folder_to_csv(root_path: str, output_csv: str):
    all_rows = []
    for file in os.listdir(root_path):
        for pdf_file in glob.glob(os.path.join(root_path, file)):
            try:
                all_rows.extend(extract_CK_Tangerine_transactions_from_pdf(pdf_file, "Tangerine"))
            except Exception as e:
                print(f"Error processing {pdf_file}: {e}")
                continue

    df = pd.DataFrame(all_rows, columns=["TRANS DATE", "DESCRIPTION", "AMOUNT", "CARD TYPE"])
    df.to_csv(output_csv, index=False)
    print(f"Wrote {len(df)} rows to {output_csv}")


def extract_TD_folder_to_csv(root_path: str, output_csv: str):
    all_rows = []
    for file in os.listdir(root_path):
        for pdf_file in glob.glob(os.path.join(root_path, file)):
            try:
                all_rows.extend(extract_TD_transactions_from_pdf(pdf_file))
            except Exception as e:
                print(f"Error processing {pdf_file}: {e}")
                continue
    df = pd.DataFrame(all_rows, columns=["TRANS DATE", "POSTING DATE", "DESCRIPTION", "AMOUNT", "CARD TYPE"])
    df.to_csv(output_csv, index=False)
    print(f"Wrote {len(df)} rows to {output_csv}")
    

In [140]:

if __name__ == "__main__":
    # 1) install deps:c
    #    pip install pdfplumber pandas openpyxl
    #
    # 2) run on a folder of PDFs (edit paths as needed):
    # root_LOC_path = r"C:\Users\linhl\OneDrive\Desktop\Consultation SDV - D2V Science Inc\Tax 2025\BMO\LOC"
    # extract_LOC_folder_to_csv(
    #         root_path=  root_LOC_path,
    #         output_csv=r"C:\Users\linhl\OneDrive\Desktop\Consultation SDV - D2V Science Inc\Tax Extractor\output\BMO_LOC_transactions.csv",
    #     )
    
    # root_CK_path = r"C:\Users\linhl\OneDrive\Desktop\Consultation SDV - D2V Science Inc\Tax 2025\BMO\CK"
    # extract_CK_folder_to_csv(
    #         root_path=  root_CK_path,
    #         output_csv=r"C:\Users\linhl\OneDrive\Desktop\Consultation SDV - D2V Science Inc\Tax Extractor\output\BMO_CK_transactions.csv",
    #     )

    # root_Tangerine_path = r"C:\Users\linhl\OneDrive\Desktop\Consultation SDV - D2V Science Inc\Tax 2025\Tangerine"
    # extract_Tangerine_folder_to_csv(
    #         root_path=  root_Tangerine_path,
    #         output_csv=r"C:\Users\linhl\OneDrive\Desktop\Consultation SDV - D2V Science Inc\Tax Extractor\output\Tangerine_transactions.csv",
    #     )
    
    root_TD_path = r"C:\Users\linhl\OneDrive\Desktop\Consultation SDV - D2V Science Inc\Tax 2025\TD"
    extract_TD_folder_to_csv(
            root_path=  root_TD_path,
            output_csv=r"C:\Users\linhl\OneDrive\Desktop\Consultation SDV - D2V Science Inc\Tax Extractor\output\TD_transactions.csv",
        )

    
    
    


Error processing C:\Users\linhl\OneDrive\Desktop\Consultation SDV - D2V Science Inc\Tax 2025\TD\Aug-Sep2025.csv: No /Root object! - Is this really a PDF?
Error processing C:\Users\linhl\OneDrive\Desktop\Consultation SDV - D2V Science Inc\Tax 2025\TD\Dec-Jan2026.csv: No /Root object! - Is this really a PDF?
Error processing C:\Users\linhl\OneDrive\Desktop\Consultation SDV - D2V Science Inc\Tax 2025\TD\Nov-Dec2025.csv: No /Root object! - Is this really a PDF?
Error processing C:\Users\linhl\OneDrive\Desktop\Consultation SDV - D2V Science Inc\Tax 2025\TD\Oct-Nov2025.csv: No /Root object! - Is this really a PDF?
Error processing C:\Users\linhl\OneDrive\Desktop\Consultation SDV - D2V Science Inc\Tax 2025\TD\Sep-Oct2025.csv: No /Root object! - Is this really a PDF?
Wrote 735 rows to C:\Users\linhl\OneDrive\Desktop\Consultation SDV - D2V Science Inc\Tax Extractor\output\TD_transactions.csv


In [118]:
tmp = ""

print(is_td_transaction_line("1avr 3avr 60,00$"))

True
